# Lab 4.1: Explaining Phishing Detectors

AI for Cybersecurity (D7084E / D7041E), Group 6
Kirill Silchenko (kirsil-5@student.ltu.se) and Stefanos Ntentopoulos (stente-5@student.ltu.se)

Parts A-D of the lab in order: train the detectors (supervised, semi-supervised, self-supervised),
explain them (glass boxes, SHAP, LIME), test the detector and the explanation (ablation, LIME
stability), and build a Belief Rule-Based expert system from the two most important features.
`random_state=42` throughout. Run from top to bottom; the dataset is `data/dataset_phishing.csv`
(or `dataset_phishing.csv` uploaded next to the notebook in Colab).

*Generated by `tools/assemble.py` from the notebooks in `parts/`. Do not edit this file by hand.*

<!-- STEP A0 -->
# Setup: lab steps A0–A3

Owner: T3. Every part notebook starts with `%run 00_setup.ipynb`; this notebook loads the data,
makes the 60 / 20 / 20 split and defines `report()`. The first line of every cell is a step marker
used by `tools/assemble.py`.

**A0. Libraries.** Colab has pandas, scikit-learn and matplotlib but not the two explanation
libraries. Locally everything comes from `requirements.txt`, so nothing is installed.

In [1]:
# STEP A0
# Install shap and lime only when they are missing (Colab). Locally they come from requirements.txt.
import importlib.util

if importlib.util.find_spec("shap") is None or importlib.util.find_spec("lime") is None:
    %pip install -q shap lime

<!-- STEP A1 -->
## Step A1: Load the data

Web Page Phishing Detection (Hannousse & Yahiouche, 2021), CC BY 4.0: 11,430 URLs, half phishing
and half legitimate, 87 features; the label is in `status`. See `data/README.md`.

In [2]:
# STEP A1
import os
from pathlib import Path

import numpy as np
import pandas as pd

# The part notebooks live in parts/. Work from the repository root, so that
# data/ and results/ mean the same thing in every notebook.
if Path.cwd().name == "parts":
    os.chdir("..")

DATA = Path("data/dataset_phishing.csv")
if not DATA.exists():
    DATA = Path("dataset_phishing.csv")   # Colab: uploaded with the folder icon

# Every step saves its figures and tables here (TASKLIST section 2.5)
for folder in ["results/figures", "results/tables"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)                     # the file you downloaded
print(df.shape)                            # should print (11430, 89)
print(df["status"].value_counts())         # 5715 phishing, 5715 legitimate

assert df.shape == (11430, 89), "unexpected dataset shape"
assert (df["status"].value_counts() == 5715).all(), "classes are not 5715 / 5715"
df.head()

(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


<!-- STEP A2 -->
## Step A2: Prepare and split

The label becomes 1 = phishing, 0 = legitimate, and the raw `url` text is dropped. We split
60 / 20 / 20: **training** to learn, **validation** to choose settings, **test** only for the final
scores. `stratify` keeps 50% phishing in each part.

In [3]:
# STEP A2
from sklearn.model_selection import train_test_split

# The label: 1 = phishing, 0 = legitimate
y = (df["status"] == "phishing").astype(int)

# The features: everything except the raw URL text and the label.
# .astype(float) avoids type warnings later when we change values.
X = df.drop(columns=["url", "status"]).astype(float)

# 60% train, 20% validation, 20% test, same phishing share in each part
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(len(X_train), len(X_val), len(X_test))   # 6858 2286 2286

assert X.shape[1] == 87, "expected 87 features"
assert (len(X_train), len(X_val), len(X_test)) == (6858, 2286, 2286)
for name, part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:10s} phishing share: {part.mean():.3f}")
    assert abs(part.mean() - 0.5) < 0.001, f"{name} is not 50% phishing"

6858 2286 2286
train      phishing share: 0.500
validation phishing share: 0.500
test       phishing share: 0.500


<!-- STEP A3 -->
## Step A3: A helper to score any model

Macro-F1, recall, ROC-AUC and FAR (false-alarm rate: share of legitimate URLs wrongly flagged).
Every model in the lab is scored with this function.

In [4]:
# STEP A3
from sklearn.metrics import (f1_score, recall_score, roc_auc_score,
                             confusion_matrix)

def report(model, X, y):
    """The four numbers we use for every model in this lab (3 decimals)."""
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    scores = {
        "macro_F1": f1_score(y, pred, average="macro"),
        "recall":   recall_score(y, pred),     # share of phishing we caught
        "ROC_AUC":  roc_auc_score(y, prob),
        "FAR":      fp / (fp + tn),            # share of legit wrongly flagged
    }
    return {k: round(float(v), 3) for k, v in scores.items()}

<!-- STEP A7 -->
### 5.2 Semi-supervised: 5% labels plus unlabelled URLs

## Step A7: Hide 95% of the labels, and train the lower lines

We split the **training** set into 5% labelled and 95% unlabelled URLs, and train a forest and a
logistic regression on the 5% only. These are the **lower lines**: the unlabelled data only helped if a
model beats them. `y_hidden` is kept only to check the guesses in step A8: never train on it.

In [5]:
# STEP A7
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Pretend that only 5% of the training URLs have a label
X_lab, X_unlab, y_lab, y_hidden = train_test_split(
    X_train, y_train, train_size=0.05, stratify=y_train, random_state=42)
print(len(X_lab), "labelled,", len(X_unlab), "unlabelled")
# y_hidden = the labels we pretend we do not have. NEVER train on it:
# it is only used in step A8 to count how many pseudo-labels were right.

# The 5% split comes from the training set only: no validation or test URL is in it
assert (len(X_lab), len(X_unlab)) == (342, 6516)
assert set(X_lab.index) | set(X_unlab.index) == set(X_train.index)
assert not set(X_train.index) & (set(X_val.index) | set(X_test.index))
print("phishing share in the 5%:", round(y_lab.mean(), 3))

# Two lower lines: the same models, trained on the few labels only
rf_few = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_few.fit(X_lab, y_lab)
print("forest, 5% labels:", report(rf_few, X_val, y_val))

logreg_few = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000)),
])
logreg_few.fit(X_lab, y_lab)
print("logreg, 5% labels:", report(logreg_few, X_val, y_val))

342 labelled, 6516 unlabelled
phishing share in the 5%: 0.5


forest, 5% labels: {'macro_F1': 0.938, 'recall': 0.946, 'ROC_AUC': 0.984, 'FAR': 0.069}


/home/steven/Desktop/AI-for-Cybersecurity-Lab4.1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


logreg, 5% labels: {'macro_F1': 0.912, 'recall': 0.915, 'ROC_AUC': 0.962, 'FAR': 0.092}


<!-- STEP A8 -->
## Step A8: Pseudo-labelling

As in Lab 2: the forest guesses the unlabelled URLs, keeps only the guesses it is very sure about
(`CUTOFF = 0.9`), and adds them as if they were labels. Three rounds, then the final model.
Each round we also count how many of the added guesses were right, by comparing them with
`y_hidden`. That is the only use of `y_hidden`: it never reaches `fit()`.

In [6]:
# STEP A8
# Semi-supervised: pseudo-labelling (the same idea as Lab 2)
X_now, y_now = X_lab.copy(), y_lab.copy()          # starts with the 5% real labels
pool = X_unlab.copy()                              # URLs without a label
rf_semi = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
CUTOFF = 0.9                                       # try 0.85 or 0.95 too

rounds = []                                        # one row per round, for the report
for rnd in [1, 2, 3]:
    rf_semi.fit(X_now, y_now)
    p = rf_semi.predict_proba(pool)[:, 1]
    sure = (p >= CUTOFF) | (1 - p >= CUTOFF)       # sure phishing OR sure legit
    guess = pd.Series((p[sure] >= 0.5).astype(int), index=pool.index[sure])
    X_now = pd.concat([X_now, pool[sure]])         # add them as labels
    y_now = pd.concat([y_now, guess])
    pool = pool[~sure]
    print(f"round {rnd}: added {sure.sum()} guesses,",
          f"{len(pool)} still unlabelled")

    # Check only: how many of this round's guesses were right? (y_hidden is not used to train)
    right = (guess == y_hidden.loc[guess.index])
    rounds.append({"round": rnd, "added": int(sure.sum()),
                   "added_phishing": int(guess.sum()),
                   "added_legitimate": int((guess == 0).sum()),
                   "correct": int(right.sum()),
                   "accuracy": round(float(right.mean()), 4) if len(guess) else float("nan"),
                   "still_unlabelled": len(pool)})

rf_semi.fit(X_now, y_now)                          # final model: labels + guesses
print("semi-supervised:", report(rf_semi, X_val, y_val))

rounds = pd.DataFrame(rounds)
all_guesses = y_now.iloc[len(y_lab):]              # every pseudo-label, without the 5% real ones
print(f"training rows for the final model: {len(X_now)} "
      f"({len(y_lab)} real labels + {len(all_guesses)} guesses)")
print("accuracy of all guesses:",
      round(float((all_guesses == y_hidden.loc[all_guesses.index]).mean()), 4))
rounds.to_csv("results/tables/A8_pseudo_label_rounds.csv", index=False)
rounds

round 1: added 2778 guesses, 3738 still unlabelled


round 2: added 1122 guesses, 2616 still unlabelled


round 3: added 365 guesses, 2251 still unlabelled


semi-supervised: {'macro_F1': 0.931, 'recall': 0.946, 'ROC_AUC': 0.982, 'FAR': 0.083}
training rows for the final model: 4607 (342 real labels + 4265 guesses)
accuracy of all guesses: 0.9885


,round,added,added_phishing,added_legitimate,correct,accuracy,still_unlabelled
0,1,2778,1280,1498,2762,0.9942,3738
1,2,1122,720,402,1105,0.9848,2616
2,3,365,213,152,349,0.9562,2251


<!-- STEP A9 -->
### 5.3 Self-supervised: learn from the URLs alone first

## Step A9: A fill-in-the-blanks network, then 5% labels

We hide 20% of the (scaled) values and train a small network to give back the complete row, without
labels. Its middle layer has 32 numbers per URL. A logistic regression then uses those 32 numbers and
the 5% labels. The fair comparison is `logreg_few` from step A7: the same model on the same 342 labels,
but on the original 87 features.

In [7]:
# STEP A9
# Self-supervised: learn from the URLs alone, no labels (like Lab 3)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import FunctionTransformer

scaler = StandardScaler().fit(X_train)            # uses no labels
Z = scaler.transform(X_train)

# The made-up task: hide 20% of the clues and ask the network to fill them in
rng = np.random.default_rng(42)
hidden_part = rng.random(Z.shape) < 0.2
Z_masked = np.where(hidden_part, 0.0, Z)           # 0 = average value after scaling

ae = MLPRegressor(hidden_layer_sizes=(32,), max_iter=300, random_state=42)
ae.fit(Z_masked, Z)                                # input: with holes, target: complete
print("fill-in error:", round(ae.loss_, 3))

def encode(X):
    """The 32 numbers the network learned for each URL (its middle layer)."""
    h = scaler.transform(X) @ ae.coefs_[0] + ae.intercepts_[0]
    return np.maximum(h, 0)                        # same "relu" as inside the network

# Now use the 5% labels, but on the 32 learned numbers
self_sup = Pipeline([
    ("encode", FunctionTransformer(encode)),
    ("lr", LogisticRegression(max_iter=1000)),
])
self_sup.fit(X_lab, y_lab)
print("self-supervised:", report(self_sup, X_val, y_val))
print("logreg, 5% labels (the fair comparison):", report(logreg_few, X_val, y_val))

fill-in error: 0.167


/home/steven/Desktop/AI-for-Cybersecurity-Lab4.1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


self-supervised: {'macro_F1': 0.914, 'recall': 0.918, 'ROC_AUC': 0.966, 'FAR': 0.09}
logreg, 5% labels (the fair comparison): {'macro_F1': 0.912, 'recall': 0.915, 'ROC_AUC': 0.962, 'FAR': 0.092}
